# Subject Independant 1D-CNN + LSTM With Attention

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Set hardware device allocation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using execution device: {device}")

# Global parameters matching dataset specifications
WINDOW_SIZE = 250   # 250 samples = 250 ms window
STEP_SIZE = 50      # 50 samples step size = 200 samples overlap
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 20

# Define paths (Update this string to match your local folder setup)
LABELED_DIR = Path("D:/EMG/EMG_Large/sEMG-dataset/filtered/csv_labeled")

Using execution device: cuda


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

# Global Configuration Parameters
WINDOW_SIZE = 250
STEP_SIZE = 50
BATCH_SIZE = 128
NUM_CLASSES = 10
LABELED_DIR = Path("./filtered/csv_labeled").resolve()

def load_1d_normalized_subject_split(labeled_dir: Path, window_size=250, step_size=50):
    all_files = sorted(labeled_dir.glob("*.csv"))
    if not all_files:
        raise FileNotFoundError(f"No CSV files found in {labeled_dir}")
        
    train_files = all_files[:32]   # Subjects 1-32
    val_files   = all_files[32:36] # Subjects 33-36
    test_files  = all_files[36:40] # Subjects 37-40
    
    def process_files_to_1d(file_list):
        X_list, y_list = [], []
        
        for file_path in file_list:
            print(f"Extracting & Normalizing: {file_path.name}")
            df = pd.read_csv(file_path, header=None)
            
            raw_emg = df.iloc[:, 0:4].values.astype(np.float32)
            labels_column = df.iloc[:, 4].values
            
            # ✅ FIX 1: Per-subject, per-channel Z-score on raw EMG
            mean_emg = np.mean(raw_emg, axis=0)   # shape (4,)
            std_emg  = np.std(raw_emg, axis=0)    # shape (4,)
            std_emg[std_emg == 0] = 1e-8
            normalized_emg = (raw_emg - mean_emg) / std_emg
            
            data = np.hstack((normalized_emg, labels_column.reshape(-1, 1)))
            
            # Sliding window extraction
            num_windows = (len(data) - window_size) // step_size + 1
            for i in range(num_windows):
                start = i * step_size
                end   = start + window_size
                
                # ✅ FIX 2: Check ALL rows in window, not just first/last
                window_labels = data[start:end, 4]
                if len(np.unique(window_labels)) != 1:
                    continue  # skip transition windows

                # Shape: (250, 4) → will transpose to (4, 250) for BiLSTM later
                X_list.append(data[start:end, 0:4])
                y_list.append(int(data[start, 4]))
                    
        return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

    print("--- Extracting 1D Time-Series Training Sets ---")
    X_train, y_train = process_files_to_1d(train_files)
    
    print("\n--- Extracting 1D Time-Series Validation Sets ---")
    X_val, y_val = process_files_to_1d(val_files)
    
    print("\n--- Extracting 1D Time-Series Testing Sets ---")
    X_test, y_test = process_files_to_1d(test_files)

    # ✅ FIX 3: Correct label indexing (must be 0-9, not 1-10)
    print("\nRaw unique labels in train:", np.unique(y_train))
    if y_train.min() == 1:
        print("Labels are 1-indexed! Shifting to 0-indexed...")
        y_train -= 1
        y_val   -= 1
        y_test  -= 1
    print("Corrected unique labels:", np.unique(y_train))

    # ✅ FIX 4: Global normalization across windows using TRAIN stats only
    # X shape here is (N, 250, 4) — normalize across N and timesteps, per channel
    global_mean = X_train.mean(axis=(0, 1), keepdims=True)  # shape (1, 1, 4)
    global_std  = X_train.std(axis=(0, 1), keepdims=True)   # shape (1, 1, 4)
    global_std[global_std == 0] = 1e-8

    X_train = (X_train - global_mean) / global_std
    X_val   = (X_val   - global_mean) / global_std  # ← train stats, NOT val's own
    X_test  = (X_test  - global_mean) / global_std  # ← train stats, NOT test's own

    # ✅ Sanity checks
    print("\n--- Sanity Checks ---")
    print(f"X_train shape : {X_train.shape}  | dtype: {X_train.dtype}")
    print(f"X_val shape   : {X_val.shape}  | dtype: {X_val.dtype}")
    print(f"X_test shape  : {X_test.shape}  | dtype: {X_test.dtype}")
    print(f"y_train range : {y_train.min()} to {y_train.max()}")
    print(f"X_train mean  : {X_train.mean():.4f} (should be ~0)")
    print(f"X_train std   : {X_train.std():.4f}  (should be ~1)")
    print(f"\nTrain class dist : {dict(sorted(Counter(y_train.tolist()).items()))}")
    print(f"Val class dist   : {dict(sorted(Counter(y_val.tolist()).items()))}")

    return X_train, y_train, X_val, y_val, X_test, y_test



In [3]:
# Load the data arrays
X_train, y_train, X_val, y_val, X_test, y_test = load_1d_normalized_subject_split(LABELED_DIR, WINDOW_SIZE, STEP_SIZE)

# Convert arrays directly into streaming PyTorch DataLoaders
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds  = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nFinal Feature Shapes (Batch size ready):")
print(f"Train Dataset Tensor Shape: {X_train.shape}")  # (Samples, 250, 4)
print(f"Val Dataset Tensor Shape:   {X_val.shape}")    # (Samples, 250, 4)
print(f"Test Dataset Tensor Shape:  {X_test.shape}")   # (Samples, 250, 4)

--- Extracting 1D Time-Series Training Sets ---
Extracting & Normalizing: 10_filtered.csv
Extracting & Normalizing: 11_filtered.csv
Extracting & Normalizing: 12_filtered.csv
Extracting & Normalizing: 13_filtered.csv
Extracting & Normalizing: 14_filtered.csv
Extracting & Normalizing: 15_filtered.csv
Extracting & Normalizing: 16_filtered.csv
Extracting & Normalizing: 17_filtered.csv
Extracting & Normalizing: 18_filtered.csv
Extracting & Normalizing: 19_filtered.csv
Extracting & Normalizing: 1_filtered.csv
Extracting & Normalizing: 20_filtered.csv
Extracting & Normalizing: 21_filtered.csv
Extracting & Normalizing: 22_filtered.csv
Extracting & Normalizing: 23_filtered.csv
Extracting & Normalizing: 24_filtered.csv
Extracting & Normalizing: 25_filtered.csv
Extracting & Normalizing: 26_filtered.csv
Extracting & Normalizing: 27_filtered.csv
Extracting & Normalizing: 28_filtered.csv
Extracting & Normalizing: 29_filtered.csv
Extracting & Normalizing: 2_filtered.csv
Extracting & Normalizing: 30_f

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class CNNBiLSTMAttention(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=128, num_classes=10):
        super(CNNBiLSTMAttention, self).__init__()
        
        # ✅ Stage 1: 1D CNN — extracts local temporal waveform features
        # Input: (batch, 250, 4) → need (batch, 4, 250) for Conv1d
        self.cnn = nn.Sequential(
            # Block 1: detect short bursts and sharp activations
            nn.Conv1d(in_channels=4, out_channels=32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),          # (batch, 32, 125)

            # Block 2: detect medium-scale patterns across channels
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),          # (batch, 64, 62)

            # Block 3: detect higher-level gesture-specific patterns
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
                                                  # (batch, 128, 62)
        )
        
        # ✅ Stage 2: BiLSTM — models temporal dynamics of CNN features
        # Input to LSTM: (batch, 62, 128) — 62 timesteps, 128 features each
        self.lstm = nn.LSTM(
            input_size=128,          # CNN output channels
            hidden_size=hidden_dim,  # 128
            num_layers=2,            # stacked for deeper temporal modeling
            bidirectional=True,
            batch_first=True,
            dropout=0.3              # dropout between LSTM layers
        )
        
        # ✅ Stage 3: Attention — focus on gesture-relevant timesteps
        # BiLSTM output: (batch, 62, hidden_dim * 2 = 256)
        self.attention_linear = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.attention_query  = nn.Linear(hidden_dim * 2, 1, bias=False)
        
        # ✅ Stage 4: Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (batch, 250, 4)
        
        # Stage 1: CNN needs (batch, channels, time) = (batch, 4, 250)
        x = x.permute(0, 2, 1)                        # → (batch, 4, 250)
        x = self.cnn(x)                                # → (batch, 128, 62)
        
        # Stage 2: LSTM needs (batch, time, features) = (batch, 62, 128)
        x = x.permute(0, 2, 1)                        # → (batch, 62, 128)
        lstm_out, _ = self.lstm(x)                     # → (batch, 62, 256)
        
        # Stage 3: Attention over 62 timesteps
        energy           = torch.tanh(self.attention_linear(lstm_out))  # (batch, 62, 256)
        attention_scores = self.attention_query(energy)                  # (batch, 62, 1)
        attention_weights = F.softmax(attention_scores, dim=1)           # (batch, 62, 1)
        context_vector   = torch.sum(attention_weights * lstm_out, dim=1) # (batch, 256)
        
        # Stage 4: Classify
        return self.classifier(context_vector)


# Build model
model = CNNBiLSTMAttention(input_dim=4, hidden_dim=128, num_classes=10).to(device)
print(model)

# Quick shape check
dummy = torch.randn(8, 250, 4).to(device)
out   = model(dummy)
print(f"Output shape: {out.shape}")  # should be (8, 10)

CNNBiLSTMAttention(
  (cnn): Sequential(
    (0): Conv1d(4, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention_linear): Linear(in_features=256, out_features=256, bias=True)
  (attention_query): Linear(in_features=256, out_features=1, bias=False)
  (classifier): Sequentia

In [7]:
# Instantiate model on available processing hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNBiLSTMAttention(input_dim=4, hidden_dim=64, num_classes=NUM_CLASSES).to(device)
print(model)

CNNBiLSTMAttention(
  (cnn): Sequential(
    (0): Conv1d(4, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (lstm): LSTM(128, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention_linear): Linear(in_features=128, out_features=128, bias=True)
  (attention_query): Linear(in_features=128, out_features=1, bias=False)
  (classifier): Sequential

In [8]:

# ── DataLoaders ────────────────────────────────────────────────────────────────
# X shape: (N, 250, 4) — already correct for BiLSTM (batch, timesteps, channels)
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_dataset   = TensorDataset(torch.tensor(X_val),   torch.tensor(y_val))
test_dataset  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Class-weighted loss (handles any imbalance) ────────────────────────────────
counts       = np.bincount(y_train)
weights      = 1.0 / counts.astype(np.float32)
weights      = weights / weights.sum() * NUM_CLASSES
class_weights = torch.FloatTensor(weights).to(device)
criterion    = nn.CrossEntropyLoss(weight=class_weights)

# ── Optimizer + Scheduler ──────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5, verbose=True
)

# ── Prediction distribution diagnostic ────────────────────────────────────────
def check_pred_distribution(model, loader, device, label=""):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch_X, _ in loader:
            preds = model(batch_X.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
    print(f"  [{label}] Pred dist: {dict(sorted(Counter(all_preds).items()))}")




Train batches : 6350 | Val batches: 794


c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [12]:
from tqdm import tqdm

In [13]:
# ── Training Loop ──────────────────────────────────────────────────────────────
best_val_acc    = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):

    # --- TRAINING PHASE ---
    model.train()
    running_loss  = 0.0
    correct_train = 0
    total_train   = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]", unit="batch")

    for batch_X, batch_y in train_bar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss    = criterion(outputs, batch_y)
        loss.backward()

        # Gradient clipping — prevents exploding gradients in LSTM
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss  += loss.item() * batch_X.size(0)
        _, predicted   = torch.max(outputs, 1)
        total_train   += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc":  f"{(predicted == batch_y).sum().item() / batch_y.size(0) * 100:.2f}%",
            "lr":   f"{optimizer.param_groups[0]['lr']:.6f}"
        })

    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc  = (correct_train / total_train) * 100

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss    = 0.0
    correct_val = 0
    total_val   = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Val  ]", unit="batch", leave=False)

    with torch.no_grad():
        for batch_X, batch_y in val_bar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss    = criterion(outputs, batch_y)

            val_loss    += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val   += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()

            val_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc  = (correct_val / total_val) * 100

    # Step scheduler on val accuracy
    scheduler.step(epoch_val_acc)

    # Epoch summary
    print(f"Summary -> Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}% | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Prediction distribution every 5 epochs
    if (epoch + 1) % 5 == 0:
        print("  Diagnostics:")
        check_pred_distribution(model, train_loader, device, "Train")
        check_pred_distribution(model, val_loader,   device, "Val  ")

    # Save best model + early stopping
    if epoch_val_acc > best_val_acc:
        best_val_acc     = epoch_val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_bilstm_emg.pth")
        print(f"--> ✅ Saved best model! Val Acc: {best_val_acc:.2f}%")
    else:
        patience_counter += 1
        print(f"    No improvement. Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n⛔ Early stopping at epoch {epoch+1}!")
        break

    print("-" * 80)

# Always restore best weights
model.load_state_dict(torch.load("best_bilstm_emg.pth"))
print(f"\n✅ Training complete! Best Val Acc: {best_val_acc:.2f}%")

Epoch 01/20 [Train]: 100%|██████████| 6350/6350 [02:53<00:00, 36.60batch/s, loss=1.6509, acc=28.91%, lr=0.001000]


Summary -> Train Loss: 1.8202 | Train Acc: 27.94% | Val Loss: 1.9747 | Val Acc: 24.93% | LR: 0.001000
--> ✅ Saved best model! Val Acc: 24.93%
--------------------------------------------------------------------------------


Epoch 02/20 [Train]: 100%|██████████| 6350/6350 [02:47<00:00, 37.88batch/s, loss=1.6370, acc=28.91%, lr=0.001000]


Summary -> Train Loss: 1.7023 | Train Acc: 31.45% | Val Loss: 2.1277 | Val Acc: 24.68% | LR: 0.001000
    No improvement. Patience: 1/10
--------------------------------------------------------------------------------


Epoch 03/20 [Train]: 100%|██████████| 6350/6350 [02:59<00:00, 35.37batch/s, loss=1.6303, acc=36.72%, lr=0.001000]


...

Summary -> Train Loss: 1.4413 | Train Acc: 40.76% | Val Loss: 2.4314 | Val Acc: 23.27% | LR: 0.000250
    No improvement. Patience: 8/10
--------------------------------------------------------------------------------


Epoch 10/20 [Train]: 100%|██████████| 6350/6350 [02:47<00:00, 38.00batch/s, loss=1.3556, acc=48.44%, lr=0.000250]


Summary -> Train Loss: 1.3991 | Train Acc: 42.36% | Val Loss: 2.4372 | Val Acc: 23.61% | LR: 0.000250
  Diagnostics:
  [Train] Pred dist: {0: 96360, 1: 77686, 2: 81718, 3: 62242, 4: 80491, 5: 63524, 6: 107521, 7: 80234, 8: 95095, 9: 67929}
  [Val  ] Pred dist: {0: 9115, 1: 10715, 2: 13345, 3: 10925, 4: 11997, 5: 7575, 6: 7380, 7: 5334, 8: 17460, 9: 7754}
    No improvement. Patience: 9/10
--------------------------------------------------------------------------------


Epoch 11/20 [Train]: 100%|██████████| 6350/6350 [02:47<00:00, 37.91batch/s, loss=1.4234, acc=51.56%, lr=0.000250]
                                                                                      

Summary -> Train Loss: 1.3892 | Train Acc: 42.93% | Val Loss: 2.4357 | Val Acc: 23.99% | LR: 0.000250
    No improvement. Patience: 10/10

⛔ Early stopping at epoch 11!

✅ Training complete! Best Val Acc: 24.93%
